### Q1: Install Spark and PySpark

In [1]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types

In [2]:
spark = SparkSession.builder \
    .master("local[*]") \
    .appName('test') \
    .getOrCreate()

In [3]:
pyspark.__file__

'/usr/local/spark/python/pyspark/__init__.py'

In [4]:
spark.version

'3.5.0'

In [5]:
# Скачать файл
# !wget https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2024-10.parquet

In [6]:
!ls -lh /home/jovyan/work/hw/yellow_tripdata_2024-10.parquet

-rwxrwxr-x 1 jovyan users 62M Dec 18 21:21 /home/jovyan/work/hw/yellow_tripdata_2024-10.parquet


In [7]:
df = spark.read.parquet("yellow_tripdata_2024-10.parquet")
df.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)



Answer: **3.5.0**

### Q2: Yellow October 2024

In [8]:
df \
    .repartition(4) \
    .write \
    .mode("overwrite") \
    .parquet('data/pq/yellow/2024/10/')

In [10]:
df.schema

StructType([StructField('VendorID', IntegerType(), True), StructField('tpep_pickup_datetime', TimestampNTZType(), True), StructField('tpep_dropoff_datetime', TimestampNTZType(), True), StructField('passenger_count', LongType(), True), StructField('trip_distance', DoubleType(), True), StructField('RatecodeID', LongType(), True), StructField('store_and_fwd_flag', StringType(), True), StructField('PULocationID', IntegerType(), True), StructField('DOLocationID', IntegerType(), True), StructField('payment_type', LongType(), True), StructField('fare_amount', DoubleType(), True), StructField('extra', DoubleType(), True), StructField('mta_tax', DoubleType(), True), StructField('tip_amount', DoubleType(), True), StructField('tolls_amount', DoubleType(), True), StructField('improvement_surcharge', DoubleType(), True), StructField('total_amount', DoubleType(), True), StructField('congestion_surcharge', DoubleType(), True), StructField('Airport_fee', DoubleType(), True)])

In [9]:
!ls -lh /home/jovyan/work/hw/data/pq/yellow/2024/10/

total 90M
-rw-r--r-- 1 jovyan users 23M Mar  6 15:16 part-00000-a69a19c7-9946-47be-a98d-6769cef764f2-c000.snappy.parquet
-rw-r--r-- 1 jovyan users 23M Mar  6 15:16 part-00001-a69a19c7-9946-47be-a98d-6769cef764f2-c000.snappy.parquet
-rw-r--r-- 1 jovyan users 23M Mar  6 15:16 part-00002-a69a19c7-9946-47be-a98d-6769cef764f2-c000.snappy.parquet
-rw-r--r-- 1 jovyan users 23M Mar  6 15:16 part-00003-a69a19c7-9946-47be-a98d-6769cef764f2-c000.snappy.parquet
-rw-r--r-- 1 jovyan users   0 Mar  6 15:16 _SUCCESS


Answer: **23M**

### Q3: How many taxi trips were there on Oktober 15?

In [11]:
from pyspark.sql import functions as F

In [12]:
df_yellow = spark.read.parquet('data/pq/yellow/2024/10/')

In [16]:
df_yellow.head()

Row(VendorID=1, tpep_pickup_datetime=datetime.datetime(2024, 10, 3, 3, 40, 19), tpep_dropoff_datetime=datetime.datetime(2024, 10, 3, 3, 46, 11), passenger_count=1, trip_distance=0.6, RatecodeID=1, store_and_fwd_flag='N', PULocationID=48, DOLocationID=161, payment_type=1, fare_amount=6.5, extra=3.5, mta_tax=0.5, tip_amount=2.9, tolls_amount=0.0, improvement_surcharge=1.0, total_amount=14.4, congestion_surcharge=2.5, Airport_fee=0.0)

In [20]:
df_yellow \
    .withColumn('pickup_date', F.to_date("tpep_pickup_datetime")) \
    .filter(F.to_date("tpep_pickup_datetime") == F.lit('2024-10-15')) \
    .count()

128893

In [26]:
df_yellow.select("tpep_pickup_datetime", "VendorID", "passenger_count", "trip_distance").show()

+--------------------+--------+---------------+-------------+
|tpep_pickup_datetime|VendorID|passenger_count|trip_distance|
+--------------------+--------+---------------+-------------+
| 2024-10-03 03:40:19|       1|              1|          0.6|
| 2024-10-10 14:08:03|       2|              1|         1.12|
| 2024-10-05 23:46:46|       1|              2|          8.2|
| 2024-10-09 11:26:13|       2|              1|         1.45|
| 2024-10-10 14:09:32|       1|              1|          0.9|
| 2024-10-01 15:47:00|       2|              1|         1.08|
| 2024-10-02 11:17:03|       1|              2|          1.3|
| 2024-10-02 07:40:58|       2|              4|         2.59|
| 2024-10-09 18:55:48|       2|              1|         1.93|
| 2024-10-08 17:00:53|       1|              2|          4.9|
| 2024-10-07 12:35:00|       2|              1|         1.32|
| 2024-10-01 10:49:05|       2|              1|         1.92|
| 2024-10-03 02:55:56|       1|              2|          1.6|
| 2024-1

In [32]:
df_yellow.explain()

== Physical Plan ==
*(1) ColumnarToRow
+- FileScan parquet [VendorID#57,tpep_pickup_datetime#58,tpep_dropoff_datetime#59,passenger_count#60L,trip_distance#61,RatecodeID#62L,store_and_fwd_flag#63,PULocationID#64,DOLocationID#65,payment_type#66L,fare_amount#67,extra#68,mta_tax#69,tip_amount#70,tolls_amount#71,improvement_surcharge#72,total_amount#73,congestion_surcharge#74,Airport_fee#75] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/home/jovyan/work/hw/data/pq/yellow/2024/10], PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen...




In [34]:
df_yellow.createOrReplaceTempView('yellow_tripdata_2024_10')

In [35]:
spark.sql("""
SELECT
    COUNT(1)
FROM 
    yellow_tripdata_2024_10
WHERE
    to_date(tpep_pickup_datetime) = '2024-10-15';
""").show()

+--------+
|count(1)|
+--------+
|  128893|
+--------+



Answer: **128893**

### Q4: Longest trip for each day

In [37]:
df_yellow.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'Airport_fee']

In [43]:
df_yellow \
    .withColumn(
        'duration', 
        (F.unix_timestamp('tpep_dropoff_datetime') - F.unix_timestamp('tpep_pickup_datetime')) / 3600) \
    .withColumn(
        'pickup_date', 
        F.to_date(F.col('tpep_pickup_datetime'))
    ) \
    .groupBy('pickup_date') \
    .agg(
        F.round(F.max('duration'), 1).alias('max_duration')  # Заміна назви 'max(duration)' на щось більш стандартне
    ) \
    .orderBy(F.col('max_duration').desc()) \
    .limit(5) \
    .show()

+-----------+------------+
|pickup_date|max_duration|
+-----------+------------+
| 2024-10-16|       162.6|
| 2024-10-03|       143.3|
| 2024-10-22|       137.8|
| 2024-10-18|       114.8|
| 2024-10-21|        89.9|
+-----------+------------+



In [44]:
spark.sql("""
SELECT
    to_date(tpep_pickup_datetime) AS pickup_date,
    round(MAX((unix_timestamp(tpep_dropoff_datetime) - unix_timestamp(tpep_pickup_datetime)) / 3600), 1) AS duration
FROM 
    yellow_tripdata_2024_10
GROUP BY
    to_date(tpep_pickup_datetime)
ORDER BY
    duration DESC
LIMIT 5;
""").show()

+-----------+--------+
|pickup_date|duration|
+-----------+--------+
| 2024-10-16|   162.6|
| 2024-10-03|   143.3|
| 2024-10-22|   137.8|
| 2024-10-18|   114.8|
| 2024-10-21|    89.9|
+-----------+--------+



### Q6: Least frequent pickup location zone

In [51]:
# Скачать файл
!wget https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv

--2025-03-06 17:07:55--  https://d37ci6vzurychx.cloudfront.net/misc/taxi_zone_lookup.csv
Resolving d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)... 52.84.111.169, 52.84.111.52, 52.84.111.148, ...
Connecting to d37ci6vzurychx.cloudfront.net (d37ci6vzurychx.cloudfront.net)|52.84.111.169|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 12331 (12K) [text/csv]
Saving to: ‘taxi_zone_lookup.csv.1’

taxi_zone_lookup.cs 100%[===================>]  12.04K  --.-KB/s    in 0.001s  

2025-03-06 17:07:55 (13.4 MB/s) - ‘taxi_zone_lookup.csv.1’ saved [12331/12331]



In [49]:
# Показать первые строки файла
!head taxi_zone_lookup.csv

"LocationID","Borough","Zone","service_zone"
1,"EWR","Newark Airport","EWR"
2,"Queens","Jamaica Bay","Boro Zone"
3,"Bronx","Allerton/Pelham Gardens","Boro Zone"
4,"Manhattan","Alphabet City","Yellow Zone"
5,"Staten Island","Arden Heights","Boro Zone"
6,"Staten Island","Arrochar/Fort Wadsworth","Boro Zone"
7,"Queens","Astoria","Boro Zone"
8,"Queens","Astoria Park","Boro Zone"
9,"Queens","Auburndale","Boro Zone"


In [50]:
# Прочитать файл в DataFrame с использованием Spark
df1 = spark.read \
.option("header", "true") \
.csv('taxi_zone_lookup.csv')

# Показать уникальные значения в столбце 'Zone'
df1.select('Zone').distinct().show()

# Показать содержимое DataFrame
df1.show()

# Подсчитать количество строк в DataFrame
df1.count()

+--------------------+
|                Zone|
+--------------------+
|Governor's Island...|
|           Homecrest|
|              Corona|
|    Bensonhurst West|
|         Westerleigh|
|      Newark Airport|
|Charleston/Totten...|
|          Douglaston|
|East Concourse/Co...|
|          Mount Hope|
|      Pelham Parkway|
|         Marble Hill|
|           Rego Park|
|       Dyker Heights|
|Heartland Village...|
|Upper East Side S...|
|   Kew Gardens Hills|
|       Rikers Island|
|             Bayside|
|     Jackson Heights|
+--------------------+
only showing top 20 rows

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Z

265

In [52]:
# Записать DataFrame в формате parquet
df1.write.parquet('zones')

In [53]:
!ls -lh zones

total 8.0K
-rw-r--r-- 1 jovyan users 5.8K Mar  6 17:09 part-00000-0599fa19-8b3e-414f-9a29-f87732649c4b-c000.snappy.parquet
-rw-r--r-- 1 jovyan users    0 Mar  6 17:09 _SUCCESS


In [54]:
df_zones = spark.read.parquet('zones')

df_zones.columns

['LocationID', 'Borough', 'Zone', 'service_zone']

In [55]:
df_zones.createOrReplaceTempView('zones')

In [59]:
spark.sql("""
SELECT
    zones.Zone AS zone,
    COUNT(1) AS trip_count
FROM
    yellow_tripdata_2024_10
JOIN
    zones
ON
    yellow_tripdata_2024_10.PULocationID = zones.LocationID
GROUP BY
        zone
ORDER BY
        trip_count
LIMIT 1;
""").show()

+--------------------+----------+
|                zone|trip_count|
+--------------------+----------+
|Governor's Island...|         1|
+--------------------+----------+

